# Tiền xử lý văn bản tiếng Việt
> **Mục đích:** Hate Speech Detection  
> **Các bước:** Chuẩn hóa unicode → lowercase -> chuẩn hóa dấu câu -> chuẩn hóa câu

## 1. Import thư viện

In [1]:
import re
import unicodedata
import string
import pandas as pd
from pathlib import Path

## 2. Dữ liệu dùng chung
### 2.1 Bảng nguyên âm & Stopwords

In [2]:
# Bảng nguyên âm tiếng Việt (kiểu "hoá" → "hóa")
VOWELS_MAP = {
    "òa": "oà", "óa": "oá", "ỏa": "oả", "õa": "oã", "ọa": "oạ",
    "òe": "oè", "óe": "oé", "ỏe": "oẻ", "õe": "oẽ", "ọe": "oẹ",
    "ùy": "uỳ", "úy": "uý", "ủy": "uỷ", "ũy": "uỹ", "ụy": "uỵ",
}

### 2.4 Patterns: Emoji & ký tự không hợp lệ

In [3]:
EMOJI_RANGES = [
    (0x1F600, 0x1F64F), (0x1F300, 0x1F5FF), (0x1F680, 0x1F6FF),
    (0x1F700, 0x1F77F), (0x1F780, 0x1F7FF), (0x1F800, 0x1F8FF),
    (0x1F900, 0x1F9FF), (0x1FA00, 0x1FA6F), (0x1FA70, 0x1FAFF),
    (0x2600,  0x26FF),  (0x2700,  0x27BF),  (0xFE00,  0xFE0F),
    (0x1F1E0, 0x1F1FF), (0x200D,  0x200D),  (0x20E3,  0x20E3),
]

EMOJI_PATTERN = re.compile(
    "[" + "".join(f"\\U{lo:08X}-\\U{hi:08X}" for lo, hi in EMOJI_RANGES) + "]",
    flags=re.UNICODE,
)

VALID_CHARS_PATTERN = re.compile(
    r"[^\w\s\u00C0-\u024F\u1E00-\u1EFF\u0300-\u036F]", flags=re.UNICODE
)

## 3. Các hàm xử lý

In [4]:
def normalize_unicode(text: str) -> str:
    """Bước a: Chuẩn hóa unicode về NFC."""
    return unicodedata.normalize("NFC", text)


def to_lowercase(text: str) -> str:
    """Bước b: Chuyển về chữ thường."""
    return text.lower()

def normalize_tone_marks(text: str) -> str:
    """Bước c: Chuẩn hóa dấu thanh tiếng Việt (hoà → hòa)."""
    for wrong, correct in VOWELS_MAP.items():
        text = text.replace(wrong, correct)
    return text

def remove_punctuation(text: str) -> str:
    """Loại bỏ dấu câu, thay bằng khoảng trắng."""
    punct = string.punctuation + "`"
    return text.translate(str.maketrans(punct, " " * len(punct)))


def remove_invalid_chars(text: str) -> str:
    """Xóa các ký tự không hợp lệ trong văn bản tiếng Việt."""
    return VALID_CHARS_PATTERN.sub(" ", text)


def remove_emoji(text: str) -> str:
    """Xóa các emoji."""
    return EMOJI_PATTERN.sub(" ", text)

def normalize_whitespace(text: str) -> str:
    """Loại bỏ khoảng trắng thừa."""
    return re.sub(r"\s+", " ", text).strip()

## 4. Pipeline tiền xử lý

In [5]:
def normalize_sentence(text: str) -> str:
    """
    Bước d: Chuẩn hóa câu.
    Thứ tự: emoji → ký tự không hợp lệ → dấu câu → whitespace
    """
    text = remove_emoji(text)
    text = remove_invalid_chars(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)
    return text


def preprocess(text: str) -> str:
    """Pipeline đầy đủ: a → b → c → d."""
    if not isinstance(text, str) or not text.strip():
        return ""
    text = normalize_unicode(text)        #a
    text = to_lowercase(text)             # b
    text = normalize_tone_marks(text)     # c
    text = normalize_sentence(text)       # d
    return text

## 5. Kiểm tra với ví dụ mẫu

In [6]:
test_cases = [
    "Thằng đó vcl, ngu vl!!!",
    "sp này dc k bạn ơi? 😊",
    "v4i l0n, đồ sv!!!",
    "mn ơi, ad rep ib mk nha 😂",
    "Con này hoà nhã lắm nha",
]

print(f"{'Gốc':<40} {'Sau xử lý':<40}")
print("-" * 80)
for text in test_cases:
    result = preprocess(text)
    print(f"{text:<40} {result:<40}")

Gốc                                      Sau xử lý                               
--------------------------------------------------------------------------------
Thằng đó vcl, ngu vl!!!                  thằng đó vcl ngu vl                     
sp này dc k bạn ơi? 😊                    sp này dc k bạn ơi                      
v4i l0n, đồ sv!!!                        v4i l0n đồ sv                           
mn ơi, ad rep ib mk nha 😂                mn ơi ad rep ib mk nha                  
Con này hoà nhã lắm nha                  con này hoà nhã lắm nha                 


## 6. Xử lý file Excel

In [7]:
def process_excel(
    input_path: str,
    text_column: str = "text",
    output_path: str = None
) -> pd.DataFrame:
    """
    Đọc file Excel, tiền xử lý cột văn bản, lưu kết quả.

    Args:
        input_path:   Đường dẫn file Excel đầu vào (.xlsx).
        text_column:  Tên cột chứa văn bản (mặc định: 'text').
        output_path:  Đường dẫn file đầu ra. Nếu None, tự tạo tên.

    Returns:
        DataFrame đã được tiền xử lý.
    """
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Không tìm thấy file: {input_path}")

    df = pd.read_excel(input_path)

    if text_column not in df.columns:
        raise ValueError(
            f"Không tìm thấy cột '{text_column}'. "
            f"Các cột hiện có: {df.columns.tolist()}"
        )

    df["text_original"] = df[text_column].astype(str)
    df[text_column] = df["text_original"].apply(preprocess)

    if output_path is None:
        output_path = input_path.parent / f"{input_path.stem}_preprocessed.xlsx"

    df.to_excel(output_path, index=False)
    print(f"✅ Đã lưu kết quả vào: {output_path}")
    print(f"   Số dòng đã xử lý: {len(df)}")
    return df

## 7. Chạy trên file Excel
> Thay `input_file` bằng đường dẫn thực tế của bạn.

In [8]:
from google.colab import files

print("Vui lòng tải lên file .xlsx của bạn:")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f"User uploaded file \'{fn}\'")
  # Assuming you want to use the first uploaded file
  input_file = fn
  break


Vui lòng tải lên file .xlsx của bạn:


Saving data_tho.xlsx to data_tho.xlsx
User uploaded file 'data_tho.xlsx'


In [9]:
# ── Cấu hình ──────────────────────────────
input_file  = "data_tho.xlsx"   # Đường dẫn file đầu vào (được đặt từ ô tải lên)
text_col    = "text"                # Tên cột văn bản
output_file = None                  # None → tự tạo tên
# ───────────────────────────────────────────

df = process_excel(input_file, text_col, output_file)

# Xem vài dòng kết quả
print("\n── Ví dụ trước/sau tiền xử lý ──")
for i in range(min(5, len(df))):
    print(f"\n[{i+1}] Gốc : {df['text_original'].iloc[i]}")
    print(f"     Sau : {df['text'].iloc[i]}")

✅ Đã lưu kết quả vào: data_tho_preprocessed.xlsx
   Số dòng đã xử lý: 71051

── Ví dụ trước/sau tiền xử lý ──

[1] Gốc : m bựa à
     Sau : m bựa à

[2] Gốc : mấy ảnh xạo á bây
     Sau : mấy ảnh xạo á bây

[3] Gốc : cái này là bè nhóm, chứ không phải bạn. không phải là không đua theo không nổi, nhưng mà mệt lắm, không đáng giao du với đám bè nhóm này. lờ c nó đi.
     Sau : cái này là bè nhóm chứ không phải bạn không phải là không đua theo không nổi nhưng mà mệt lắm không đáng giao du với đám bè nhóm này lờ c nó đi

[4] Gốc : ôi xời, toàn bọn sĩ diện hão, bạn kệ mịe bno đi
     Sau : ôi xời toàn bọn sĩ diện hão bạn kệ mịe bno đi

[5] Gốc : vấn đề thể lực thôi, hài ỉa mấy con vợ "tâm lý chiến" với chả "trí tuệ nữ thấp hơn" =)))
     Sau : vấn đề thể lực thôi hài ỉa mấy con vợ tâm lý chiến với chả trí tuệ nữ thấp hơn
